# Chapter 04: Reasoning about GPT architecture size


Before allocating a model, parameter arithmetic reveals where capacity
and memory go. These exercises derive the dominant attention and
feed-forward terms, then estimate complete GPT-2-style model sizes from
configuration values.


    This notebook is an independently written practice companion. It
    does not reproduce the book's prose, figures, data, or solutions.
    Read the [Chinese chapter](https://github.com/skindhu/Build-A-Large-Language-Model-CN/blob/main/cn-Book/4.%E4%BB%8E%E9%9B%B6%E5%BC%80%E5%A7%8B%E5%AE%9E%E7%8E%B0%E4%B8%80%E4%B8%AA%E7%94%A8%E4%BA%8E%E6%96%87%E6%9C%AC%E7%94%9F%E6%88%90%E7%9A%84%20GPT%20%E6%A8%A1%E5%9E%8B.md) for the complete narrative
    and exercise statements, and use the
    [canonical MIT companion code](https://github.com/rasbt/LLMs-from-scratch)
    as an additional reference.

    **Prerequisites:** Python 3.10+ and PyTorch 2.2+. Every example is
    synthetic, deterministic, CPU-friendly, and makes no network call.

    **Workflow**

    1. Read the concept and trace the shapes by hand.
    2. Replace one `LLM_TODO` and its `NotImplementedError`.
    3. Run that exercise's check cell.
    4. Explain the result before moving to the next exercise.
    5. Compare with `solution.py` only after making a serious attempt.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int
    context_length: int
    embedding_dim: int
    num_layers: int
    num_heads: int

## Exercise 4.1: Count attention and feed-forward parameters

    **Objective.** Derive parameter counts from layer dimensions rather than instantiating modules.


A dense projection from $D_{in}$ to $D_{out}$ has
$D_{in}D_{out}$ weights and $D_{out}$ biases. A standard attention block
has four width-preserving projections: query, key, value, and output.
Its leading weight term is therefore $4D^2$.

A feed-forward network expands from $D$ to $ED$ and contracts back.
Its leading weight term is $D(ED)+(ED)D=2ED^2$. With expansion $E=4$,
this is $8D^2$, approximately twice attention's projection parameters.
Bias terms matter for exact counts but not for the dominant ratio.


    ### Data-flow walkthrough


For width `D=768`, Q/K/V/output each use a `768 x 768` weight. The FFN
uses `768 x 3072` and `3072 x 768`. Return exact totals including the
stated biases, then compare the ratio instead of relying on memory.


    ### Hints and common mistakes


Count parameters, not activations. Do not multiply by sequence length or
batch size; shared weights are reused at every position. State whether
biases are included so another implementation can reproduce the count.

In [ ]:
def projection_parameter_counts(
    embedding_dim: int, expansion: int = 4
) -> dict[str, int]:
    # LLM_TODO_CH04_01_COUNTS
    raise NotImplementedError(
        "count exact attention and FFN weights plus biases"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
counts = projection_parameter_counts(768)
assert set(counts) == {"attention", "feed_forward"}
ratio = counts["feed_forward"] / counts["attention"]
assert 1.9 < ratio < 2.1
print(f"Exercise 4.1 passed: FFN/attention = {ratio:.3f}")

## Exercise 4.2: Estimate complete GPT-2 variants

    **Objective.** Compute tied-output parameter and weights-only memory estimates without allocation.


A GPT-2-style estimate combines token embeddings $V\times D$, position
embeddings $T\times D$, $L$ Transformer blocks, and a final layer norm.
With pre-normalization and an expansion of four, one block contributes
approximately $12D^2$ plus linear-in-$D$ bias and normalization terms.

If the output classifier shares the token-embedding matrix, do not count
another $V\times D$ matrix. Weights-only memory is
`parameter_count * bytes_per_parameter`. Training requires substantially
more memory for gradients, optimizer state, activations, and temporary
buffers; this exercise deliberately estimates only stored weights.


    ### Data-flow walkthrough


GPT-2 small uses vocabulary 50,257, context 1,024, width 768, and 12
blocks. An XL-style configuration uses width 1,600, 48 blocks, and 25
heads. Arithmetic should place the latter near 1.6 billion parameters
without constructing a multi-gigabyte module.


    ### Hints and common mistakes


Keep units explicit: decimal billions differ from binary GiB. Validate
that width is divisible by head count even though head count does not
change the dense projection total. Avoid double-counting tied output
embeddings.

In [ ]:
def estimate_gpt2(
    config: GPTConfig, bytes_per_parameter: int = 4
) -> tuple[int, int]:
    # LLM_TODO_CH04_02_ESTIMATE
    raise NotImplementedError(
        "sum tied embeddings, blocks, and final normalization"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
small = GPTConfig(50_257, 1_024, 768, 12, 12)
xl = GPTConfig(50_257, 1_024, 1_600, 48, 25)
small_params, _ = estimate_gpt2(small)
xl_params, xl_bytes = estimate_gpt2(xl)
assert 120_000_000 < small_params < 130_000_000
assert 1_500_000_000 < xl_params < 1_650_000_000
assert xl_bytes == xl_params * 4
print(f"Exercise 4.2 passed: XL weights = {xl_bytes / 2**30:.2f} GiB")

## Chapter 04 synthesis


- Why does the FFN dominate one block's parameter count at expansion four?
- Why does changing head count leave the dense projection count unchanged?
- Which major training-memory categories are absent from weights-only memory?


    After answering these questions, run the chapter's `solution.py`
    from a terminal to compare behavior. The reference file is compact
    by design; the reasoning in this notebook is the main lesson.